In [1]:
import numpy as np
from datetime import datetime, timedelta

with open("/content/hostel_bois.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()

print("Total lines:", len(lines))

Total lines: 3178


In [2]:
messages = []
system = 0
media = 0
deleted = 0

for line in lines:

    line = line.strip()

    if line == "":
        continue

    if " - " not in line:
        continue

    first, second = line.split(" - ", 1)

    if ": " not in second:
        system += 1
        continue

    name, text = second.split(": ", 1)

    try:
        time = datetime.strptime(first, "%d/%m/%y, %H:%M")
    except:
        system += 1
        continue

    if text == "<Media omitted>":
        media += 1
        continue

    if text == "This message was deleted":
        deleted += 1
        continue

    messages.append({
        "time": time,
        "name": name,
        "text": text
    })

print("Messages:", len(messages))
print("Media:", media)
print("Deleted:", deleted)

Messages: 3127
Media: 32
Deleted: 15


In [3]:
people = []

for msg in messages:
    if msg["name"] not in people:
        people.append(msg["name"])

print("Participants:", people)
print("Number of participants:", len(people))

Participants: ['Rahul', 'Priya', 'Karan', 'Neha', 'Aman', 'Vikas']
Number of participants: 6


In [5]:
count = {}

for msg in messages:

    name = msg["name"]

    if name not in count:
        count[name] = 0

    count[name] += 1

print("GROUP OVERVIEW")
print("-" * 40)

for name in count:
    percent = count[name] / len(messages) * 100
    print(name, ":", count[name], "messages", f"({percent:.1f}%)")

GROUP OVERVIEW
----------------------------------------
Rahul : 940 messages (30.1%)
Priya : 712 messages (22.8%)
Karan : 345 messages (11.0%)
Neha : 624 messages (20.0%)
Aman : 484 messages (15.5%)
Vikas : 22 messages (0.7%)


In [6]:
dates = []

for msg in messages:
    dates.append(msg["time"].date())

start_date = min(dates)
end_date = max(dates)

print("Start date:", start_date)
print("End date:", end_date)
print("Total days:", (end_date - start_date).days + 1)

Start date: 2024-04-01
End date: 2024-05-30
Total days: 60


In [7]:
day_count = {}

for msg in messages:

    day = msg["time"].date()

    if day not in day_count:
        day_count[day] = 0

    day_count[day] += 1

busy_day = max(day_count, key=day_count.get)

print("Busiest day:", busy_day)
print("Messages:", day_count[busy_day])

Busiest day: 2024-05-04
Messages: 74


In [8]:
hour_count = {}

for msg in messages:

    hour = msg["time"].hour

    if hour not in hour_count:
        hour_count[hour] = 0

    hour_count[hour] += 1

busy_hour = max(hour_count, key=hour_count.get)

print("Busiest hour:", str(busy_hour) + ":00")
print("Messages:", hour_count[busy_hour])

Busiest hour: 18:00
Messages: 244


In [9]:
stop_words = {
    "the", "is", "a", "an", "and", "or", "to", "of",
    "in", "on", "for", "with", "this", "that", "are",
    "was", "be", "it", "i", "you", "we", "me", "my",
    "your", "our", "at", "as", "from", "have", "has",
    "had", "but", "so", "if", "just", "very"
}

word_count = {}

for msg in messages:

    text = msg["text"].lower()

    symbols = ".,!?;:\"'()[]{}<>"

    for s in symbols:
        text = text.replace(s, "")

    words = text.split()

    for word in words:

        if word in stop_words:
            continue

        if len(word) <= 1:
            continue

        if word not in word_count:
            word_count[word] = 0

        word_count[word] += 1

top_words = sorted(
    word_count.items(),
    key=lambda x: x[1],
    reverse=True
)

print("TOP 10 WORDS")
print("-" * 30)

for word, number in top_words[:10]:
    print(word, ":", number)

TOP 10 WORDS
------------------------------
how : 321
guys : 318
about : 274
hai : 268
am : 260
today : 257
he : 220
his : 217
which : 202
everyone : 187


In [10]:
for person in people:

    words = {}

    for msg in messages:

        if msg["name"] != person:
            continue

        text = msg["text"].lower()

        for s in ".,!?;:\"'()[]{}<>":
            text = text.replace(s, "")

        for word in text.split():

            if word in stop_words or len(word) <= 1:
                continue

            if word not in words:
                words[word] = 0

            words[word] += 1

    top = sorted(
        words.items(),
        key=lambda x: x[1],
        reverse=True
    )

    print("\n", person)

    for word, number in top[:5]:
        print(word, ":", number)


 Rahul
hai : 263
bhai : 159
scene : 144
kya : 133
yaar : 105

 Priya
please : 141
everyone : 137
aman : 93
anyone : 90
okay : 80

 Karan
how : 295
about : 259
his : 217
which : 202
he : 186

 Neha
am : 148
guys : 101
cant : 68
ok : 52
today : 48

 Aman
am : 98
sleep : 71
im : 56
anyone : 49
wonder : 43

 Vikas
hai : 5
haha : 4
sorry : 3
busy : 3
tha : 3


In [11]:
heatmap = np.zeros((len(people), 24), dtype=int)

for msg in messages:

    person = msg["name"]
    hour = msg["time"].hour

    row = people.index(person)

    heatmap[row][hour] += 1

print("ACTIVITY HEATMAP")
print("     ", end="")

for h in range(24):
    print(f"{h:02}", end=" ")

print()

for i in range(len(people)):

    print(f"{people[i]:<6}", end="")

    for h in range(24):

        value = heatmap[i][h]

        if value == 0:
            symbol = "."
        elif value <= 2:
            symbol = "░"
        elif value <= 5:
            symbol = "▒"
        else:
            symbol = "█"

        print(symbol, end="  ")

    print()

ACTIVITY HEATMAP
     00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23 
Rahul ▒  █  █  █  █  █  █  █  █  █  █  █  █  █  █  █  █  █  █  █  █  █  █  █  
Priya .  .  .  .  .  .  █  █  █  █  █  █  █  █  █  █  █  █  █  █  █  █  █  █  
Karan .  .  .  .  .  .  .  ▒  █  █  █  █  █  █  █  █  █  █  █  █  █  █  █  █  
Neha  .  .  .  .  .  █  ▒  █  █  █  █  █  █  █  █  █  █  █  █  █  █  █  █  █  
Aman  █  █  █  █  █  .  .  .  .  .  .  .  .  .  █  █  █  ▒  █  █  █  █  .  █  
Vikas .  .  .  .  .  .  .  ░  ░  ░  ░  .  ░  ░  .  ░  ░  ▒  ░  ░  ░  ░  ░  ░  


In [12]:
print("AVERAGE MESSAGE LENGTH")
print("-" * 30)

for person in people:

    total = 0
    number = 0

    for msg in messages:

        if msg["name"] == person:
            total += len(msg["text"].split())
            number += 1

    average = total / number

    print(person, ":", round(average, 2), "words")

AVERAGE MESSAGE LENGTH
------------------------------
Rahul : 2.55 words
Priya : 5.0 words
Karan : 57.05 words
Neha : 5.32 words
Aman : 5.02 words
Vikas : 1.82 words


In [13]:
print("NIGHT ACTIVITY")
print("-" * 30)

night_percent = {}

for person in people:

    total = 0
    night = 0

    for msg in messages:

        if msg["name"] == person:

            total += 1
            hour = msg["time"].hour

            if hour >= 23 or hour <= 4:
                night += 1

    percent = night / total * 100
    night_percent[person] = percent

    print(person, ":", round(percent, 1), "%")

NIGHT ACTIVITY
------------------------------
Rahul : 13.5 %
Priya : 1.3 %
Karan : 2.0 %
Neha : 4.8 %
Aman : 80.4 %
Vikas : 9.1 %


In [14]:
response = {}

for i in range(1, len(messages)):

    previous = messages[i - 1]
    current = messages[i]

    if previous["name"] == current["name"]:
        continue

    gap = current["time"] - previous["time"]
    minutes = gap.total_seconds() / 60

    if minutes > 0 and minutes <= 1440:

        person = current["name"]

        if person not in response:
            response[person] = []

        response[person].append(minutes)

print("RESPONSE TIME")
print("-" * 30)

average_response = {}

for person in response:

    average = sum(response[person]) / len(response[person])
    average_response[person] = average

    print(person, ":", round(average, 2), "minutes")

RESPONSE TIME
------------------------------
Priya : 43.59 minutes
Rahul : 36.76 minutes
Karan : 37.78 minutes
Neha : 42.86 minutes
Aman : 56.52 minutes
Vikas : 34.9 minutes


In [15]:
active = {}

for person in people:
    active[person] = set()

for msg in messages:
    active[msg["name"]].add(msg["time"].date())

all_days = []

day = start_date

while day <= end_date:

    all_days.append(day)
    day = day + timedelta(days=1)

silent = {}

for person in people:

    longest = 0
    current = 0

    for day in all_days:

        if day not in active[person]:

            current += 1

            if current > longest:
                longest = current

        else:
            current = 0

    silent[person] = longest

print("SILENT STREAK")
print("-" * 30)

for person in people:
    print(person, ":", silent[person], "days")

SILENT STREAK
------------------------------
Rahul : 0 days
Priya : 0 days
Karan : 0 days
Neha : 0 days
Aman : 0 days
Vikas : 11 days


In [16]:
archetype = {}

for person in people:

    if person == "Rahul":
        archetype[person] = "THE SPAMMER"

    elif person == "Priya":
        archetype[person] = "THE GROUP MOM"

    elif person == "Aman":
        archetype[person] = "THE NIGHT OWL"

    elif person == "Karan":
        archetype[person] = "THE STORYTELLER"

    elif person == "Neha":
        archetype[person] = "THE DRAMA QUEEN"

    elif person == "Vikas":
        archetype[person] = "THE GHOST"

print("PERSONALITY ARCHETYPES")
print("-" * 40)

for person in people:
    print(person, "->", archetype[person])

PERSONALITY ARCHETYPES
----------------------------------------
Rahul -> THE SPAMMER
Priya -> THE GROUP MOM
Karan -> THE STORYTELLER
Neha -> THE DRAMA QUEEN
Aman -> THE NIGHT OWL
Vikas -> THE GHOST


In [17]:
print("\n")
print("=" * 60)
print("              GROUPDNA REPORT")
print("=" * 60)

print("Group: Hostel Bois 4ever")
print("Members:", len(people))
print("Messages:", len(messages))
print("Period:", start_date, "to", end_date)

print("\nMOST ACTIVE MEMBERS")
print("-" * 40)

for person, number in sorted(
    count.items(),
    key=lambda x: x[1],
    reverse=True
):
    print(person, ":", number)

print("\nBUSIEST DAY")
print(busy_day, "-", day_count[busy_day], "messages")

print("\nBUSIEST HOUR")
print(str(busy_hour) + ":00 -", hour_count[busy_hour], "messages")

print("\nTOP WORDS")
for word, number in top_words[:10]:
    print(word, ":", number)

print("\nSILENT STREAKS")
for person in people:
    print(person, ":", silent[person], "days")

print("\nARCHETYPES")
for person in people:
    print(person, "->", archetype[person])

print("=" * 60)



              GROUPDNA REPORT
Group: Hostel Bois 4ever
Members: 6
Messages: 3127
Period: 2024-04-01 to 2024-05-30

MOST ACTIVE MEMBERS
----------------------------------------
Rahul : 940
Priya : 712
Neha : 624
Aman : 484
Karan : 345
Vikas : 22

BUSIEST DAY
2024-05-04 - 74 messages

BUSIEST HOUR
18:00 - 244 messages

TOP WORDS
how : 321
guys : 318
about : 274
hai : 268
am : 260
today : 257
he : 220
his : 217
which : 202
everyone : 187

SILENT STREAKS
Rahul : 0 days
Priya : 0 days
Karan : 0 days
Neha : 0 days
Aman : 0 days
Vikas : 11 days

ARCHETYPES
Rahul -> THE SPAMMER
Priya -> THE GROUP MOM
Karan -> THE STORYTELLER
Neha -> THE DRAMA QUEEN
Aman -> THE NIGHT OWL
Vikas -> THE GHOST
